
# Módulo 08 — Monitoramento & Diagnóstico (ML/MLOps Básico) — **Versão Comentada**
Este notebook foi montado para ensino **passo a passo**, com **explicações minuciosas em Markdown antes de cada célula**.  
O foco é **ML clássico** (scikit-learn) e os conceitos fundamentais de **observabilidade** e **MLOps**.

---
### O que você aprenderá aqui
1) **Setup e reprodutibilidade**: bibliotecas, semente aleatória.  
2) **Treino de um classificador simples**: `LogisticRegression` com `StandardScaler` em `Pipeline`.  
3) **Simulação de serving**: latência, taxa de erro, *data drift* em uma feature.  
4) **SLIs/SLOs + métricas do modelo** com janelas móveis: p95, taxa de erro, throughput, Acurácia, F1.  
5) **Drift de dados**: PSI e KS-test + visualização das distribuições.  
6) **Logs estruturados e alertas**: esquema de log, SLOs e checagem de violações.  
7) **Conclusões e próximos passos**.

> **Legenda de leitura**: Em cada seção, primeiro há um bloco **Markdown** explicando **o que o código faz e por que**, seguido do **código correspondente**.



## 🔷 Célula 1 — Setup e reprodutibilidade (explicação)
**Objetivo**: importar bibliotecas necessárias, configurar ambiente e garantir **reprodutibilidade**.

- `numpy`, `pandas`: manipulação numérica e tabular.  
- `matplotlib.pyplot`: gráficos (cada gráfico em uma figura separada).  
- `scipy.stats`: testes estatísticos (KS-test).  
- `sklearn`: dataset, *split*, *pipeline*, modelo e métricas.  
- `RANDOM_STATE = 42`: fixa a aleatoriedade (mesmos resultados a cada execução).

**Boas práticas**:
- Fixar semente é crucial para reproduzir resultados em aulas e diagnósticos.  
- Manter imports centralizados facilita organização do projeto/notebook.


In [ ]:

import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Ambiente pronto.")



## 🔷 Célula 2 — Treino rápido de um classificador (explicação)
**Objetivo**: treinar um modelo simples para termos uma **baseline** e algo para **servir**.

- **Dataset**: `load_breast_cancer()` (sklearn) — binário, bom para didática.  
- **Split**: `train_test_split(..., stratify=y, test_size=0.4)` mantém proporção das classes.  
- **Pipeline**: `StandardScaler` → `LogisticRegression` (importante para estabilizar coeficientes).  
- **Métricas em holdout**: acurácia e F1 — oferecem um ponto de referência para as janelas móveis depois.

**Por que LogisticRegression?**  
Modelo rápido, interpretável e suficiente para demonstrar *monitoramento* sem complexidades de ajuste fino.


In [ ]:

data = load_breast_cancer()
X, y = data.data, data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.4, random_state=RANDOM_STATE, stratify=y
)

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=500, random_state=RANDOM_STATE))
])
pipe.fit(X_train, y_train)

y_pred_holdout = pipe.predict(X_test)
print("Métricas em Holdout (baseline):")
print("  Acurácia:", accuracy_score(y_test, y_pred_holdout))
print("  F1:", f1_score(y_test, y_pred_holdout))



## 🔷 Célula 3 — Simulação de atendimento online (explicação)
**Objetivo**: criar um cenário semelhante a **produção**.

O que simulamos:
- **Latência** (`latency_ms`): `lognormal` com cauda longa (mais realista).  
- **Status HTTP** (`status_code`): ~2% de erros (`500`) com `np.where(cond, 500, 200)`.  
- **Data drift**: a partir de `t_drift` (60% da série), deslocamos a **feature 0** (`feat0`) somando ruído `Normal(0.8, 0.5)`.
- **Predições**: rodamos o `pipe.predict` nas entradas com drift aplicado.

**Por quê drift em uma única feature?**  
Para isolar o efeito e facilitar a **detecção** via PSI/KS e a interpretação visual.

**Saída**: construímos um `DataFrame df` com tempo `t`, latência, status, rótulo verdadeiro, predição e `feat0`.


In [ ]:

N = len(X_test)
t = np.arange(N)

# Latência e status HTTP
latency_ms = np.random.lognormal(mean=np.log(120), sigma=0.35, size=N)
status_code = np.where(np.random.rand(N) < 0.02, 500, 200)

# Drift simples numa feature (a partir de 60% do fluxo)
t_drift = int(N * 0.6)
X_test_drift = X_test.copy()
X_test_drift[t_drift:, 0] = X_test_drift[t_drift:, 0] + np.random.normal(0.8, 0.5, size=N - t_drift)

# Predições com drift aplicado
y_pred = pipe.predict(X_test_drift)

df = pd.DataFrame({
    "t": t,
    "latency_ms": latency_ms,
    "status_code": status_code,
    "y_true": y_test,
    "y_pred": y_pred,
    "feat0": X_test_drift[:, 0]
})
df.head()



## 🔷 Célula 4 — SLIs/SLOs e métricas do modelo em janela móvel (explicação)
**Objetivo**: calcular métricas **suavizadas** por **janela móvel** para reduzir ruído e destacar tendências.

- `WINDOW`: 10% de `N` (mín. 20) — regra didática.  
- `p95_latency`: quantil 95% da latência.  
- `error_rate`: fração de status `>= 500`.  
- `throughput`: **proxy** (média ~1 com ruído) — em produção, derive das contagens no tempo.  
- `acc_win`: acurácia por janela (compara `y_true` vs `y_pred`).  
- `f1_win`: F1 por janela — se a janela tiver só uma classe, retorna `NaN` para evitar métricas enganosas.

**Interpretação**: picos/saltos após `t_drift` podem indicar regressão, saturação ou drift afetando a qualidade.


In [ ]:

WINDOW = max(20, int(0.1 * N))

def rolling_p95(x, w):
    return pd.Series(x).rolling(w).quantile(0.95)

def rolling_error_rate(codes, w):
    return pd.Series(codes).rolling(w).apply(lambda s: np.mean(s >= 500))

def rolling_throughput(w):
    base = np.ones(N)
    noise = np.random.normal(0, 0.03, size=N)
    return pd.Series(base + noise).rolling(w).mean()

def rolling_acc(y_true, y_pred, w):
    return pd.Series([
        np.mean(y_true[max(0, i-w+1):i+1] == y_pred[max(0, i-w+1):i+1])
        for i in range(len(y_true))
    ])

def rolling_f1(y_true, y_pred, w):
    vals = []
    for i in range(len(y_true)):
        a = y_true[max(0, i-w+1):i+1]
        b = y_pred[max(0, i-w+1):i+1]
        # Evita erro quando só há uma classe na janela
        if len(np.unique(a)) == 2 and len(np.unique(b)) == 2:
            vals.append(f1_score(a, b))
        else:
            vals.append(np.nan)
    return pd.Series(vals)

df["p95_latency"] = rolling_p95(df["latency_ms"], WINDOW)
df["error_rate"] = rolling_error_rate(df["status_code"], WINDOW)
df["throughput"] = rolling_throughput(WINDOW)
df["acc_win"] = rolling_acc(df["y_true"].values, df["y_pred"].values, WINDOW)
df["f1_win"] = rolling_f1(df["y_true"].values, df["y_pred"].values, WINDOW)

df[["p95_latency","error_rate","throughput","acc_win","f1_win"]].tail()



## 🔷 Célula 5 — Visualização: Latência p95 (explicação)
**Objetivo**: observar a **cauda** da latência ao longo do tempo.

- Linha vertical pontilhada marca `t_drift`.  
- Aumento após `t_drift` pode sugerir regressão de performance ou saturação.

> **Dica**: em produção, monitore p95/p99 por rota e por versão de modelo.


In [ ]:

plt.figure()
df["p95_latency"].plot()
plt.axvline(x=t_drift, linestyle="--")
plt.title("Latência p95 (rolling)")
plt.xlabel("t"); plt.ylabel("ms")
plt.show()



## 🔷 Célula 6 — Visualização: Taxa de erro (explicação)
**Objetivo**: checar **confiabilidade** do serviço.

- Picos constantes de erro podem estar ligados a **intermitência** de rede, filas cheias ou falhas em dependências.  
- Após `t_drift`, observe se há **correlação** com outras métricas (ex.: F1 caindo).


In [ ]:

plt.figure()
df["error_rate"].plot()
plt.axvline(x=t_drift, linestyle="--")
plt.title("Taxa de erro (rolling)")
plt.xlabel("t"); plt.ylabel("rate")
plt.show()



## 🔷 Célula 7 — Visualização: Acurácia e F1 (explicação)
**Objetivo**: acompanhar **qualidade preditiva** no tempo.

- `acc_win` e `f1_win` suavizam oscilação e mostram tendência.  
- Quedas após `t_drift` podem indicar **drift** afetando o modelo, mesmo com infraestrutura ok.


In [ ]:

plt.figure()
df["acc_win"].plot()
df["f1_win"].plot()
plt.axvline(x=t_drift, linestyle="--")
plt.title("Acurácia e F1 (rolling)")
plt.xlabel("t"); plt.ylabel("score")
plt.show()



## 🔷 Célula 8 — Drift de dados: PSI & KS-test (explicação)
**Objetivo**: comparar distribuições **antes vs depois** de `t_drift`.

- **PSI** (Population Stability Index): mede divergência de histogramas com mesma malha de *bins*.  
  - Regra de bolso: `<0.1` estável; `0.1–0.25` atenção; `>0.25` drift relevante.  
- **KS-test**: teste não paramétrico; estatística KS e *p-value* (baixo → distribuições diferentes).

**Cuidados**:
- Escolha de *bins* do PSI influencia o valor.  
- Sempre combine **métrica + visualização** para evitar falsos positivos.


In [ ]:

def psi(ref, cur, bins=10):
    r, bin_edges = np.histogram(ref, bins=bins)
    c, _ = np.histogram(cur, bins=bin_edges)
    r = r / max(r.sum(), 1)
    c = c / max(c.sum(), 1)
    r = np.where(r == 0, 1e-6, r)
    c = np.where(c == 0, 1e-6, c)
    return np.sum((r - c) * np.log(r / c))

ref = df.loc[:t_drift-1, "feat0"].values
cur = df.loc[t_drift:, "feat0"].values

psi_val = psi(ref, cur, bins=15)
ks_stat, ks_p = stats.ks_2samp(ref, cur)

print(f"PSI: {psi_val:.3f}")
print(f"KS: stat={ks_stat:.3f}, p-value={ks_p:.3e}")



## 🔷 Célula 9 — Visualização: distribuições antes vs depois (explicação)
**Objetivo**: confirmar visualmente o drift.

- Traça **KDE** de `feat0` antes e depois.  
- Verifique se há deslocamento de média e/ou aumento de dispersão.


In [ ]:

plt.figure()
pd.Series(ref).plot(kind="kde")
pd.Series(cur).plot(kind="kde")
plt.title("Distribuição da feat0 (antes vs depois do drift)")
plt.xlabel("valor"); plt.show()



## 🔷 Célula 10 — Logs estruturados (explicação)
**Objetivo**: criar um **registro por requisição** adequado para auditoria/depuração.

- Campos essenciais: `ts`, `status`, `latency_ms`, `y_true`, `y_pred`, `feature0`, `model_version`.  
- Em produção, escreva como **JSONL** em arquivo/stream para um coletor (ELK/CloudWatch/etc.).


In [ ]:

def make_log_row(i):
    return {
        "ts": int(df.loc[i, "t"]),
        "status": int(df.loc[i, "status_code"]),
        "latency_ms": float(df.loc[i, "latency_ms"]),
        "y_true": int(df.loc[i, "y_true"]),
        "y_pred": int(df.loc[i, "y_pred"]),
        "feature0": float(df.loc[i, "feat0"]),
        "model_version": "logreg-1.0"
    }

sample_log = make_log_row(t_drift)
print(json.dumps(sample_log, ensure_ascii=False, indent=2))



## 🔷 Célula 11 — SLOs e checagem de alertas (explicação)
**Objetivo**: transformar métricas em **regras acionáveis**.

SLOs exemplares:
- `p95_latency ≤ 300 ms`  
- `error_rate ≤ 3%`  
- `f1_win ≥ 0.90`

`check_alerts(row)` avalia cada linha e aponta violações.  
**Saída**: primeiras linhas com *breaches* (se houver).


In [ ]:

SLO = {
    "p95_latency_ms_max": 300,
    "error_rate_max": 0.03,
    "f1_min": 0.90
}

def check_alerts(row):
    alerts = []
    if row.get("p95_latency", 0) > SLO["p95_latency_ms_max"]:
        alerts.append("LATENCY_P95_BREACH")
    if row.get("error_rate", 0) > SLO["error_rate_max"]:
        alerts.append("ERROR_RATE_BREACH")
    if (row.get("f1_win") is not None) and (not np.isnan(row.get("f1_win"))) and (row.get("f1_win") < SLO["f1_min"]):
        alerts.append("F1_BREACH")
    return alerts

df["alerts"] = df.apply(check_alerts, axis=1)
df[df["alerts"].apply(len) > 0].head(5)[["t","p95_latency","error_rate","f1_win","alerts"]]



## 🔷 Célula 12 — Conclusões e próximos passos (explicação)
**O que você montou**:
- **SLIs** básicos (latência, erros, throughput) e métricas de **modelo** (Acurácia/F1 em janelas).  
- **Detecção de drift** com PSI/KS + visualização.  
- **Logs estruturados** e **alertas** simples (SLO → incidente).

**Boas práticas futuras**:
- Exportar métricas em `/metrics` (Prometheus).  
- Centralizar logs (JSONL) e traçar **correlações** (latência × F1 × drift).  
- Simular **shadow/canary** com `model_version` para upgrades seguros.  
- Manter **pós-mortem** padronizado após incidentes.
